# Performing all imports
##### These are all the imports required for the project


In [18]:
# Performing all imports
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
import numpy as np

# Creating all the required datasets

In [19]:
water_quality_training = pd.read_csv('C:\EY AI & Data Challenge\Datasets\water_quality_training_dataset.csv')
terraclimate_features_training = pd.read_csv(r'C:\EY AI & Data Challenge\Datasets\terraclimate_features_training.csv')
landsat_features_training = pd.read_csv('C:\EY AI & Data Challenge\Datasets\landsat_features_training.csv')

water_quality_testing = pd.read_csv(r'C:\EY AI & Data Challenge\Datasets\submission_template.csv')
terraclimate_features_testing = pd.read_csv(r'C:\EY AI & Data Challenge\Datasets\terraclimate_features_validation.csv')
landsat_features_testing = pd.read_csv(r'C:\EY AI & Data Challenge\Datasets\landsat_features_validation.csv')



<>:1: SyntaxWarning: "\E" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\E"? A raw string is also an option.
<>:3: SyntaxWarning: "\E" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\E"? A raw string is also an option.
<>:1: SyntaxWarning: "\E" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\E"? A raw string is also an option.
<>:3: SyntaxWarning: "\E" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\E"? A raw string is also an option.
C:\Users\SW559CS\AppData\Local\Temp\ipykernel_28844\2394384508.py:1: SyntaxWarning: "\E" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\E"? A raw string is also an option.
  water_quality_training = pd.read_csv('C:\EY AI & Data Challenge\Datasets\water_quality_training_dataset.csv')
C:\Users\SW559CS\AppData\Local\Temp\ipykernel_28844\23943845

# Combining all datasets into one

In [20]:
# Combining training datasets
water_terras_df = water_quality_training.merge(
    terraclimate_features_training,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

complete_merge = water_terras_df.merge(
    landsat_features_training,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)


# Combining testing datasets
water_terras_testing_df = water_quality_testing.merge(
    terraclimate_features_testing,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

complete_merge_testing = water_terras_testing_df.merge(
    landsat_features_testing,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

complete_merge_testing.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'], inplace= True)
water_terras_testing_df.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'], inplace= True)


# Cleaning the dataset

In [21]:
# Checking for any missing values
complete_merge.isna().sum()

# Dropping all missing data
# complete_merge.dropna(inplace=True)

display(complete_merge_testing)

# Verifying that all data is of the correct datatype
complete_merge.dtypes

# Splitting datetime into individual values
def add_time_features(df, col):
    s = pd.to_datetime(df[col], format="%d-%m-%Y")
    return pd.DataFrame({
        f"{col}_year":  s.dt.year,
        f"{col}_month": s.dt.month,
        f"{col}_day":   s.dt.day,
       
    }, index=df.index)


# Converting the "Sample Data" column to the correct datatype
#complete_merge["Sample Date"] = pd.to_datetime(complete_merge["Sample Date"], format= r"%d-%m-%Y")

# Verifying that the datatypes are all correct now
complete_merge.dtypes

,Latitude,Longitude,Sample Date,pet,nir,green,swir16,swir22,NDMI,MNDWI
0,-32.043333,27.822778,01-09-2014,161.900010,15229.0,12868.0,14797.0,12421.0,0.014388,-0.069727
1,-33.329167,26.077500,16-09-2015,177.600000,NaN,NaN,NaN,NaN,NaN,NaN
2,-32.991639,27.640028,07-05-2015,158.400010,16221.0,9304.5,12536.5,9958.0,0.128123,-0.147979
3,-34.096389,24.439167,07-02-2012,130.000000,NaN,NaN,NaN,NaN,NaN,NaN
4,-32.000556,28.581667,01-10-2014,152.500000,9125.0,11100.5,9455.0,8711.0,-0.017761,0.080052
...,...,...,...,...,...,...,...,...,...,...
195,-33.771111,25.386667,06-12-2012,171.400010,17562.0,9492.0,13559.5,10235.0,0.128609,-0.176453
196,-33.185361,27.390750,04-09-2014,159.400010,15883.0,9083.5,12135.5,9484.0,0.133751,-0.143833
197,-32.043333,27.822778,28-09-2015,168.600000,13619.5,10046.5,13105.0,10969.0,0.019252,-0.132108
198,-33.001667,25.161389,08-01-2015,81.200005,13955.5,10670.0,17303.5,14835.5,-0.107105,-0.237135


Latitude                         float64
Longitude                        float64
Sample Date                          str
Total Alkalinity                 float64
Electrical Conductance           float64
Dissolved Reactive Phosphorus    float64
pet                              float64
nir                              float64
green                            float64
swir16                           float64
swir22                           float64
NDMI                             float64
MNDWI                            float64
dtype: object

# Total Alkalinity Prediction

In [22]:
# Train set
X = complete_merge.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
y = complete_merge['Total Alkalinity']

# Splitting out the date
X = X.join(add_time_features(X, "Sample Date")).drop(columns=["Sample Date"])
complete_merge_testing = complete_merge_testing.join(add_time_features(complete_merge_testing, "Sample Date")).drop(columns=["Sample Date"])

# Imputing all missing values with the mean
imputer = SimpleImputer(strategy='mean')
X = imputer.fit_transform(X)
X_test = imputer.fit_transform(complete_merge_testing)

# Ensuring that all values have been converted correctly
pd.DataFrame(X).isna().sum()
pd.DataFrame(complete_merge_testing).isna().sum()


# Train/Val Split
X_train, X_val, y_train, y_val = train_test_split(
X, y, test_size=0.2, random_state=42
)

# param_distributions = {
#     'n_estimators': np.linspace(200, 1200, num=6, dtype=int),  # [200, 400, 600, 800, 1000, 1200]
#     'max_depth': [None, 5, 10, 15, 20, 30, 50],
#     'min_samples_split': [2, 5, 10, 20],
#     'min_samples_leaf': [1, 2, 4, 8],
#     'max_features': ['sqrt', 'log2', 0.6, 0.8, 1.0],
#     'bootstrap': [True, False]
# }

# cv = KFold(n_splits=5, shuffle=True, random_state=42)

# rand_search = RandomizedSearchCV(
#     estimator=RandomForestRegressor(random_state=42),
#     param_distributions=param_distributions,
#     n_iter=50,              # increase to 100 if you have time
#     scoring='r2',
#     n_jobs=-1,
#     cv=cv,
#     verbose=3,
#     random_state=42,
#     refit=True
# )

# rand_search.fit(X_train, y_train)

# print("RandomizedSearchCV | Best CV R2:", rand_search.best_score_)
# print("RandomizedSearchCV | Best Params:", rand_search.best_params_)


In [23]:
# Defining the Model
model = RandomForestRegressor(
    n_estimators = 800,
    min_samples_split = 2,
    min_samples_leaf = 1,
    max_features = 0.8,
    max_depth = 50,
    bootstrap = True
)
model.fit(X_train, y_train)

# Generate predictions and assess performance on validation and test sets
preds_val = model.predict(X_val)
score = r2_score(preds_val, y_val)
print(f'R2 Score: {score}')
preds_alkalinity = model.predict(X_test)

# Saving results
preds_alkalinity_df = pd.DataFrame({
    'Total Alkalinity' : preds_alkalinity
})

preds_alkalinity_df.to_csv(r'C:\EY AI & Data Challenge\Datasets\Trial_1\Basic_Linear_Regression\Total Alkalinity.csv')

R2 Score: 0.8048574118724052


# Electrical Conductane Prediction

In [24]:
# Train set
X = complete_merge.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
y = complete_merge['Electrical Conductance']

# Splitting out the date
X = X.join(add_time_features(X, "Sample Date")).drop(columns=["Sample Date"])
#complete_merge_testing = complete_merge_testing.join(add_time_features(complete_merge_testing, "Sample Date")).drop(columns=["Sample Date"])

# Imputing all missing values with the mean
imputer = SimpleImputer(strategy='mean')
X = imputer.fit_transform(X)
X_test = imputer.fit_transform(complete_merge_testing)

# Ensuring that all values have been converted correctly
pd.DataFrame(X).isna().sum()
pd.DataFrame(complete_merge_testing).isna().sum()


# Train/Val Split
X_train, X_val, y_train, y_val = train_test_split(
X, y, test_size=0.2, random_state=42
)

# Hyperparameter tunining
# param_distributions = {
#     'n_estimators': np.linspace(200, 1200, num=6, dtype=int),  # [200, 400, 600, 800, 1000, 1200]
#     'max_depth': [None, 5, 10, 15, 20, 30, 50],
#     'min_samples_split': [2, 5, 10, 20],
#     'min_samples_leaf': [1, 2, 4, 8],
#     'max_features': ['sqrt', 'log2', 0.6, 0.8, 1.0],
#     'bootstrap': [True, False]
# }

# cv = KFold(n_splits=5, shuffle=True, random_state=42)

# rand_search = RandomizedSearchCV(
#     estimator=RandomForestRegressor(random_state=42),
#     param_distributions=param_distributions,
#     n_iter=50,              # increase to 100 if you have time
#     scoring='r2',
#     n_jobs=-1,
#     cv=cv,
#     verbose=1,
#     random_state=42,
#     refit=True
# )

# rand_search.fit(X_train, y_train)

# print("RandomizedSearchCV | Best CV R2:", rand_search.best_score_)
# print("RandomizedSearchCV | Best Params:", rand_search.best_params_)






In [25]:
# Defining the Model
model = RandomForestRegressor(
    n_estimators = 800,
    min_samples_split = 2,
    min_samples_leaf = 1,
    max_features = 0.8,
    max_depth = 50,
    bootstrap = True
)
model.fit(X_train, y_train)

# Generate predictions and assess performance on validation and test sets
preds_val = model.predict(X_val)
score = r2_score(preds_val, y_val)
print(f'R2 Score: {score}')
preds_ec = model.predict(X_test)

# Saving results
preds_ec_df = pd.DataFrame({
    'Electrical Conductance' : preds_ec
})

preds_ec_df.to_csv(r'C:\EY AI & Data Challenge\Datasets\Trial_1\Basic_Linear_Regression\Electrical Conductance.csv')

R2 Score: 0.8433749105695927


# Dissolved Reactive Phosphorus

In [26]:
# Train set
X = complete_merge.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
y = complete_merge['Dissolved Reactive Phosphorus']

# Splitting out the date
X = X.join(add_time_features(X, "Sample Date")).drop(columns=["Sample Date"])
#complete_merge_testing = complete_merge_testing.join(add_time_features(complete_merge_testing, "Sample Date")).drop(columns=["Sample Date"])

# Imputing all missing values with the mean
imputer = SimpleImputer(strategy='mean')
X = imputer.fit_transform(X)
X_test = imputer.fit_transform(complete_merge_testing)

# Ensuring that all values have been converted correctly
pd.DataFrame(X).isna().sum()
pd.DataFrame(complete_merge_testing).isna().sum()


# Train/Val Split
X_train, X_val, y_train, y_val = train_test_split(
X, y, test_size=0.2, random_state=42
)


# param_distributions = {
#     'n_estimators': np.linspace(200, 1200, num=6, dtype=int),  # [200, 400, 600, 800, 1000, 1200]
#     'max_depth': [None, 5, 10, 15, 20, 30, 50],
#     'min_samples_split': [2, 5, 10, 20],
#     'min_samples_leaf': [1, 2, 4, 8],
#     'max_features': ['sqrt', 'log2', 0.6, 0.8, 1.0],
#     'bootstrap': [True, False]
# }

# cv = KFold(n_splits=5, shuffle=True, random_state=42)

# rand_search = RandomizedSearchCV(
#     estimator=RandomForestRegressor(random_state=42),
#     param_distributions=param_distributions,
#     n_iter=50,              # increase to 100 if you have time
#     scoring='r2',
#     n_jobs=-1,
#     cv=cv,
#     verbose=1,
#     random_state=42,
#     refit=True
# )

# rand_search.fit(X_train, y_train)

# print("RandomizedSearchCV | Best CV R2:", rand_search.best_score_)
# print("RandomizedSearchCV | Best Params:", rand_search.best_params_)


In [27]:
# Defining the Model
model = RandomForestRegressor(
     n_estimators = 200,
    min_samples_split = 5,
    min_samples_leaf = 4,
    max_features = 0.6,
    max_depth = 15,
    bootstrap = False
)
model.fit(X_train, y_train)

# Generate predictions and assess performance on validation and test sets
preds_val = model.predict(X_val)
score = r2_score(preds_val, y_val)
print(f'R2 Score: {score}')
preds_drp = model.predict(X_test)

# Saving results
preds_drp_df = pd.DataFrame({
    'Dissolved Reactive Phosphorus' : preds_drp
})

preds_drp_df.to_csv(r'C:\EY AI & Data Challenge\Datasets\Trial_1\Basic_Linear_Regression\Dissolved Reactive Phosphorus.csv')

R2 Score: 0.5630865518440951


In [28]:
display(complete_merge_testing)

,Latitude,Longitude,pet,nir,green,swir16,swir22,NDMI,MNDWI,Sample Date_year,Sample Date_month,Sample Date_day
0,-32.043333,27.822778,161.900010,15229.0,12868.0,14797.0,12421.0,0.014388,-0.069727,2014,9,1
1,-33.329167,26.077500,177.600000,NaN,NaN,NaN,NaN,NaN,NaN,2015,9,16
2,-32.991639,27.640028,158.400010,16221.0,9304.5,12536.5,9958.0,0.128123,-0.147979,2015,5,7
3,-34.096389,24.439167,130.000000,NaN,NaN,NaN,NaN,NaN,NaN,2012,2,7
4,-32.000556,28.581667,152.500000,9125.0,11100.5,9455.0,8711.0,-0.017761,0.080052,2014,10,1
...,...,...,...,...,...,...,...,...,...,...,...,...
195,-33.771111,25.386667,171.400010,17562.0,9492.0,13559.5,10235.0,0.128609,-0.176453,2012,12,6
196,-33.185361,27.390750,159.400010,15883.0,9083.5,12135.5,9484.0,0.133751,-0.143833,2014,9,4
197,-32.043333,27.822778,168.600000,13619.5,10046.5,13105.0,10969.0,0.019252,-0.132108,2015,9,28
198,-33.001667,25.161389,81.200005,13955.5,10670.0,17303.5,14835.5,-0.107105,-0.237135,2015,1,8
